In [1]:
from dotenv import load_dotenv
from pypdf import PdfReader
from openai import OpenAI
import gradio as gr
from IPython.display import Markdown,display

In [19]:
load_dotenv(override=True)
openai = OpenAI()
ollama = OpenAI(base_url="http://localhost:11434/v1",api_key="ollama")

In [3]:
reader = PdfReader("/Users/macminived/vpvprojects/agents/1_foundations/Dev_Handson/Quantum_Theory.pdf")
qt_analysis = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        qt_analysis+=text

In [11]:
acting_role= "Physics Tutor"

role_system_prompt = f"You are a {acting_role}. You are answering questions on the website of {acting_role},\
    related to quantum theory of radiation.\
    Your responsibility is to fulfill the {acting_role} as faithfully as possible to someone who is visiting {acting_role}'s website\
    by answering the questions as correctly as possible.\
    Be very professional and mindful as if talking to a student who would be subscribing to your courses on quantum physics.\
    Do NOT assume anything. If you don't know the answer, be specific and say so."

role_system_prompt+=f"\n\n## Context:\n{qt_analysis}."
role_system_prompt+=f"With this context, stay in character as {acting_role} and chat with the user."

In [5]:
def chat_ollama(message,history):
    messages=[{"role":"system","content":role_system_prompt}]+history+[{"role":"user","content":message}]
    chatbot_model_name = "gemma3:4b"
    response = ollama.chat.completions.create(model=chatbot_model_name,messages = messages)
    answer = response.choices[0].message.content
    return answer

In [7]:
def chat_openai(message,history):
    messages=[{"role":"system","content":role_system_prompt}]+history+[{"role":"user","content":message}]
    chatbot_model_name = "gpt-5-nano"
    response = openai.chat.completions.create(model=chatbot_model_name,messages = messages)
    answer = response.choices[0].message.content
    return answer

In [12]:
gr.ChatInterface(chat_openai,type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [13]:
# Pydantic Model for evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [14]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is of acceptable quality. \
The Agent is playing the role of {acting_role} and is representing {acting_role} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential student or course buyer who came across the website. \
The Agent has been provided with context on {acting_role} in the form of theory on Quantum Radiation. Here's the information:"

evaluator_system_prompt += f"\n\n## Context:\n{qt_analysis}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [15]:
def evaluator_user_prompt(reply,message,history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [16]:
def evaluate(reply,message,history) -> Evaluation:
    messages=[{"role":"system","content":evaluator_system_prompt}]+[{"role":"user","content":evaluator_user_prompt(reply,message,history)}]
    evaluator_model_name = "gpt-5-nano"
    response = openai.chat.completions.parse(model=evaluator_model_name,messages = messages,response_format=Evaluation)
    answer = response.choices[0].message.parsed
    return answer

In [ ]:
evaluation_message = "Is Planck's thoery is based on Quantum Mechanics?"
messages = [{"role":"system", "content":role_system_prompt}]+[{"role":"user", "content":evaluation_message}]
evaluation_model_name = "gemma3:4b"
response = ollama.chat.completions.create(model=evaluation_model_name,messages=messages)
evaluation_reply = response.choices[0].message.content

In [25]:
evaluate(evaluation_reply,"Is Planck's thoery is based on Quantum Mechanics?",messages[:1])

Evaluation(is_acceptable=True, feedback='The response correctly explains that Planck’s energy quantization preceded and inspired quantum mechanics, but is not itself quantum mechanics. It provides a solid historical progression (Planck → Einstein → Bohr → development of QM) and offers further topics for deeper exploration. The tone is appropriate for a tutoring context. Minor issues: a few typographical/formatting slips (e.g., ‘hsunits’) that don’t affect understanding. Overall, a clear, accurate, and engaging answer with a good follow-up invitation.')

In [26]:
# Rerunning if the response is not acceptable

def rerun(reply,message,history,feedback):
    updated_role_system_prompt = role_system_prompt + f"\n\n## Previous answer rejected when you tried to reply\n"   
    updated_role_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_role_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_role_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
    return response.choices[0].message.content

In [27]:
def chat_updated_rerun(message,history):
    system_prompt = role_system_prompt
    messages =[{"role":"system","content":system_prompt}]+history+[{"role":"user","content":message}]
    model_name = "gemma3:4b"
    response = ollama.chat.completions.create(model=model_name,messages=messages)
    reply = response.choices[0].message.content

    evaluation = evaluate(reply,message,history)

    if evaluation.is_acceptable:
        print("Evaluation is acceptable; passed, returning reply")
    else:
        print("Evaluation is not acceptable; retrying due to the following feedback")
        print(evaluation.feedback)
        rerun(reply,message,history,evaluation.feedback)
    return reply

In [28]:
gr.ChatInterface(chat_updated_rerun,type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Evaluation is not acceptable; retrying due to the following feedback
The response is engaging and largely correct in saying Planck’s theory introduced quantization and was not based on Newton’s law. However, it contains a factual error: it states that the total energy radiated by a blackbody at a given temperature is independent of surface area. In reality, the total power emitted scales with surface area (P = σ A T^4 for a blackbody). Planck’s theory explains the spectral distribution and resolves the ultraviolet catastrophe, not this area-independence issue. It would also help to distinguish more clearly between Newton’s (or classical) laws of radiation, Rayleigh–Jeans predictions, and Stefan–Boltzmann law, and to specify that Planck’s hypothesis introduces energy quanta (E = hν). A brief correction and a bit more precise framing would improve accuracy, while staying accessible.
